# 200 Gbps benchmark: preload vs no preload

Variant of `full_run_200gbps.ipynb` that runs the `TwoHundredGbpsProcessor` twice — once with `preload=False` and once with `preload=True` — to compare the impact of coffea's branch tracing on I/O throughput. All analysis / skimming / histogramming / statistics flags are off; only the branch read pattern changes between the two runs.

## Workflow Overview

1. Setup Python path for intccms package
2. Install dependencies and register modules for cloud pickle
3. Acquire Dask client from AF environment
4. Configure parameters (disable all analysis, override branches with throughput helper)
5. Run metadata extraction (`coffea` preprocessing)
6. Run `TwoHundredGbpsProcessor` with coffea.processor.Runner

## AF flag
We might want to run this code on different facilities, which may each have their own limitations or require different dask client setups. To make it easy to switch between facilities, just set the `AF` variable to the one of your choice. If your `AF` does not exist yet, you can introduce it in this notebook in the relevant sections.

In [ ]:
AF="coffeacasa-condor" # options currently supported: [coffeacasa-condor, coffeacasa-gateway, purdue-af-k8s, purdue-af-slurm]
AUTO_CLOSE_CLIENT=False # the client setup is done with a contextmanager -- this flag decides if we automatically close the client as we exit the manager. If False, you handle closing manually. 
WARM_XCACHE=False

## Imports and dependencies

### The intccms package
The CMS implementation of the integration challenge is set in a package-like structure, which means we hae to add the source code to the python path. The package is referred to as `intccms`.

In [ ]:
# Setup Python path to include intccms package
import sys
from pathlib import Path

# Add src directory to Python path
repo_root = Path.cwd()
src_dir = repo_root / "src"
examples_dir = repo_root
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
if str(examples_dir) not in sys.path:
    sys.path.insert(0, str(examples_dir))
print(f"✅ Added {src_dir} to Python path")
print(f"✅ Added {examples_dir} to Python path")

### Installing extra dependencies
The `intccms` package requires `omegaconf` and `roastcoffea`, which is not by default on an AF. `roastcoffea` is a tool developed while working on this project and it provides an API to extract metrics from coffea-processor workflows. 

In [ ]:
try:
    import omegaconf
except ImportError:
    print("⚠️ omegaconf not found, installing...")
    ! pip install omegaconf;

try:
    import roastcoffea
except ImportError:
    print("⚠️ roastcoffea not found, installing...")
    ! pip install roastcoffea;

### Alternative coffea version
In some cases, we might need to install our own `coffea` version which is not on the AF. For example, when testing a new feature or using a recently realased version with a fix.

In [ ]:
COFFEA_VERSION = "2026.4.0"
COFFEA_PIP = f"coffea=={COFFEA_VERSION}" if "git" not in COFFEA_VERSION else COFFEA_VERSION

! pip install $COFFEA_PIP ;

# Pip-installable dependencies to install on workers
WORKER_DEPENDENCIES = [COFFEA_PIP, "roastcoffea==0.1.2"]

### Imports from stdlib and other libraries

In this notebook we use `dask` and `coffea`. 

In [ ]:
# stdlib
import cloudpickle
import copy
import os
import time

from coffea.processor import DaskExecutor, IterativeExecutor
from coffea.nanoevents import NanoAODSchema, BaseSchema

### Imports from intccms and other integration-challenge specific tooling

In [ ]:
# intccms
from intccms.schema import Config, load_config_with_restricted_cli
from intccms.utils.output import OutputDirectoryManager
from intccms.metadata_extractor import DatasetMetadataManager
from intccms.datasets import DatasetManager
from intccms.analysis import run_processor_workflow, TwoHundredGbpsProcessor
from intccms.utils.tools import get_branches_for_fraction, warm_xcache

# roastcoffea metrics
from roastcoffea import MetricsCollector

### Registering packages with cloudpickle
The intccms cannot be installed on the workers via `pip`, and the configuration files are in python modules which also cannot be installed on the workers. So we need to register them with `cloudpickle` to allow dask to serialize them and send them out.

In [ ]:
import intccms
import example_cms_200gbps

# Register modules for cloud pickle
cloudpickle.register_pickle_by_value(intccms)
cloudpickle.register_pickle_by_value(example_cms_200gbps)

## Dask client setup

This notebook uses the `DaskExecutor` from `coffea` to distribute the task graph on the AF. The client setup varies in different facilities, so we implement a function which returns the correct client. The function does so by providing a context manager, within which the client is alive.

In [ ]:
from intccms.utils.dask_client import acquire_client, live_prints

## Configuration Setup

Same configuration loading as `full_run.ipynb`, but with all analysis/skimming/histogramming/statistics turned off. The branches config is overridden with `get_branches_for_fraction` to control what fraction of each file gets read.

In [ ]:
# intccms configuration import
from example_cms_200gbps.configs.configuration import config as original_config

# Create a deepcopy that we can manipulate
config = copy.deepcopy(original_config)

# Limit files for testing
config["datasets"]["max_files"] = None # None would run over all availale files

# Skip known-bad files
config["datasets"]["skip_files"] = [
    "92D0BDF3-91AE-514F-88B5-8F591450B8AD.root",
    "8E2613E5-9327-D644-9567-C3A5CE721D27.root"
]

# Use local output directory
config["general"]["output_dir"] = "example_cms_200gbps/outputs/"

# Preprocessing (coffea) can be executed once and results loaded
config["general"]["run_metadata_generation"] = True

# Processor: only read branches, no analysis or skimming
config["general"]["run_processor"] = True
config["general"]["run_analysis"] = False
config["general"]["save_skimmed_output"] = False
config["general"]["run_histogramming"] = False
config["general"]["run_systematics"] = False
config["general"]["run_corrections"] = False
config["general"]["run_statistics"] = False

# ---------------------------------------------------------------------------
# Override branches: pick the biggest branches covering TARGET_FRACTION of
# the file. Set cache_path so subsequent runs skip the slow measurement step.
#
# If you have a representative data file, pass data_file= to split MC-only
# branches into mc_branches automatically.
# ---------------------------------------------------------------------------
TARGET_FRACTION = 0.04  # fraction of file to read (1.0 = everything)

SAMPLE_MC_FILE = (
    "root://xcache//store/mc/RunIISummer20UL16NanoAODv9/ZPrimeToTT_M2000_W200_TuneCP2_13TeV-madgraph-pythia8/NANOAODSIM/106X_mcRun2_asymptotic_v17-v2/2530000/288B512F-09A1-5D48-8D1B-6216C5904FB5.root"
)
SAMPLE_DATA_FILE = (
    "root://xcache//store/data/Run2016C/SingleMuon/NANOAOD/HIPM_UL2016_MiniAODv2_NanoAODv9-v2/40000/1D381615-0139-A540-AC3C-B3BC7C2B781F.root"
)
BRANCH_CACHE = "example_cms/configs/branch_sizes.json"

branches, mc_branches = get_branches_for_fraction(
    SAMPLE_MC_FILE,
    target_fraction=TARGET_FRACTION,
    cache_path=BRANCH_CACHE,
    data_file=SAMPLE_DATA_FILE,
    veto=("LHEPdfWeight","GenPart", "GenJet", "TrigObj", "LHEPart"),
)
config["preprocess"]["branches"] = branches
config["preprocess"]["mc_branches"] = mc_branches

print(f"Branches: {sum(len(v) for v in branches.values())} fields across {len(branches)} collections")
print(f"MC-only branches: {sum(len(v) for v in mc_branches.values())} fields")

print(branches, "\n", mc_branches)

cli_args = []
full_config = load_config_with_restricted_cli(config, cli_args)
validated_config = Config(**full_config)

## Running the Workflow

Same steps as `full_run.ipynb`:

1. Setting up output directories
2. Building an input dataset manager
3. Running or loading the coffea preprocessing
4. Run the throughput processor (instead of the analysis processor)

### Output manager setup

In [ ]:
output_manager = OutputDirectoryManager(
    root_output_dir=validated_config.general.output_dir,
    cache_dir=validated_config.general.cache_dir,
    metadata_dir=validated_config.general.metadata_dir,
    skimmed_dir=validated_config.general.skimmed_dir
)

### Input dataset manager setup

In [ ]:
dataset_manager = DatasetManager(validated_config.datasets)

### Coffea preprocessing

In [ ]:
metadata_generator = DatasetMetadataManager(
  dataset_manager=dataset_manager,
  output_manager=output_manager,
  config=validated_config,
)

if metadata_generator.generate_metadata:
  with acquire_client(AF, close_after=AUTO_CLOSE_CLIENT, pip_packages=WORKER_DEPENDENCIES) as (client, cluster):
      metadata_generator.run(executor=DaskExecutor(client=client))
else:
  metadata_generator.run()  # No client needed

# Build metadata lookup and extract workitems
metadata_lookup = metadata_generator.build_metadata_lookup()
workitems = metadata_generator.workitems
;

### Run Throughput Processor

In [ ]:
# Create the throughput processor
processor = TwoHundredGbpsProcessor(
    config=validated_config,
    output_manager=output_manager,
    metadata_lookup=metadata_lookup,
)

for prl in [True, False]:
    # Run processor workflow with roastcoffea metrics collection
    with acquire_client(AF, close_after=AUTO_CLOSE_CLIENT, pip_packages=WORKER_DEPENDENCIES, profile_output_dir=f"{output_manager.root_output_dir}/profiling/", profile_suffix="200gbps_preload",) as (client, cluster):
        t0 = time.perf_counter()
        output, report = run_processor_workflow(
            config=validated_config,
            output_manager=output_manager,
            metadata_lookup=metadata_lookup,
            processor=processor,
            workitems=workitems,
            executor=DaskExecutor(client=client, treereduction=8, retries=0),
            schema=BaseSchema,
            preload=prl,
        )
        t1 = time.perf_counter()

    wall_time = t1 - t0
    print(f"Done in {wall_time:.1f} seconds (preload = {prl})")
    print(f"Total events processed: {output.get('processed_events', 0):,}")